# Lesson 02 - Utforska Microsoft Agent Framework

The **Microsoft Agent Framework (MAF)** är ett enhetligt ramverk för att bygga AI-agenter. Det tillhandahåller en ren, sammansättbar arkitektur med fyra kärnkomponenter:

- **Client** – ansluter till en AI-modellendpoint och hanterar kommunikation
- **Agent** – omsluter en klient med instruktioner och verktygsdefinitioner
- **Tools** – utökar agentens kapabiliteter med anpassade funktioner som modellen kan anropa
- **Session** – underhåller konversationshistorik för flerstegsinteraktioner

I denna lektion ska vi bygga en **resebokningsagent** som kontrollerar destinations tillgänglighet med hjälp av dessa koncept.


## Installation


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Förstå agentramverksarkitekturen

Microsoft Agent Framework följer en lagerindelad arkitektur:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Klient** – En `AzureAIProjectAgentProvider` ansluter till en Azure OpenAI-distribution. Den hanterar autentisering, förfrågningsformatering och svarstolkning.
2. **Agent** – Skapas från klienten via `provider.create_agent()`, agenten kombinerar modellåtkomst med instruktioner (systemprompt) och verktyg.
3. **Verktyg** – Python-funktioner dekorerade med `@tool` som agenten kan anropa för att utföra åtgärder eller hämta data.
4. **Session** – Ett `AgentSession`-objekt (skapat via `agent.create_session()`) som lagrar konversationshistorik och möjliggör flerstegsdialog där agenten minns tidigare sammanhang.

Låt oss bygga varje lager steg för steg.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Lägga till verktyg med @tool-dekoratorn

Verktyg låter agenter utföra åtgärder utöver att generera text. `@tool`-dekoratorn omvandlar en vanlig Python-funktion till något som agenten kan anropa.

Viktiga punkter:
- Använd `Annotated[type, "description"]` så att modellen förstår varje parameter.
- Docstringen blir verktygsbeskrivningen som modellen ser.
- `approval_mode="never_require"` betyder att verktyget körs automatiskt utan användarbekräftelse.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Skapa en agent med verktyg

Nu kombinerar vi klienten, instruktionerna och verktygen till en agent. `instructions` fungerar som systemprompten — de definierar agentens personlighet och beteende.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Fleromgångssamtal med sessioner

En `AgentSession` (skapas via `agent.create_session()`) håller reda på alla meddelanden i en konversation. Genom att skicka samma session till varje `agent.run()`-anrop har agenten tillgång till hela konversationshistoriken och kan referera tillbaka till tidigare meddelanden.

Vi skickar `tools=[check_destination_availability]` så att agenten kan använda vår tillgänglighetskontroll vid varje omgång.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Sammanfattning

I denna lektion utforskade du de fyra pelarna i Microsoft Agent Framework:

| Concept | Vad du lärde dig |
|---------|------------------|
| **Client** | `AzureAIProjectAgentProvider` ansluter till Azure OpenAI med autentisering baserad på referenser |
| **Agent** | `provider.create_agent()` paketera en modellanslutning med instruktioner och ett namn |
| **Tools** | `@tool`-dekorationen exponerar Python-funktioner för agenten att anropa |
| **Session** | `agent.create_session()` upprätthåller samtalshistorik över flera vändningar |

Dessa byggstenar sätts samman för att skapa agenter som kan hålla naturliga konversationer, anropa externa funktioner och bibehålla kontext — grunden för mer avancerade agentmönster i senare lektioner.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, vänligen var medveten om att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess modersmål bör betraktas som den auktoritativa källan. För viktig information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för eventuella missförstånd eller feltolkningar som uppstår från användningen av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
